# 申鹤 RVC 语音 API
在 Colab 上运行，提供申鹤（中文 CV）语音转换 API。

### 使用步骤
1. 点击 代码执行程序 → 全部运行
2. 等待安装完成（约 5 分钟）
3. 看到 ngrok URL 后，复制到本地 agent 的 config 中

In [ ]:
# 1. 安装依赖
!pip install -q torch torchaudio fairseq librosa soundfile pyworld praat-parselmouth numpy scipy flask flask-cors pyngrok huggingface_hub
!pip install -q ffmpeg-python av
!apt-get update -qq && apt-get install -y -qq ffmpeg ngrok 2>/dev/null
print('✅ 依赖安装完成')

In [ ]:
# 2. 克隆 RVC 代码
!git clone --depth 1 https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git /content/rvc
import sys, os
RVC_ROOT = '/content/rvc'
sys.path.insert(0, RVC_ROOT)
sys.path.insert(0, f'{RVC_ROOT}/infer/lib')
sys.path.insert(0, f'{RVC_ROOT}/infer/modules')
os.chdir(RVC_ROOT)
print('✅ RVC 代码就绪')

In [ ]:
# 3. 下载申鹤模型 + HuBERT
from huggingface_hub import hf_hub_download
import os

MODEL_DIR = '/content/models'
os.makedirs(MODEL_DIR, exist_ok=True)
os.environ['weight_root'] = MODEL_DIR
os.environ['index_root'] = MODEL_DIR

# 下载中文申鹤 RVC 模型 (Cinnamomo)
print('下载申鹤模型...')
for fname in ['genshin_impact/shenhe_e250_s11250.pth', 'genshin_impact/shenhe_pitch.index']:
    path = hf_hub_download('Cinnamomo/rvc_models', filename=fname, local_dir=MODEL_DIR)
    print(f'  ✅ {fname}')

# 下载 HuBERT base 模型
print('下载 HuBERT...')
hf_hub_download('lj1995/VoiceConversionWebUI', filename='hubert_base.pt', local_dir='/content/rvc/assets/hubert')
print('✅ 模型就绪')

In [ ]:
# 4. 初始化 RVC 推理
import torch
from infer.modules.vc.modules import VC

class RVCConfig:
    def __init__(self):
        self.is_half = True  # GPU 用 fp16
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.n_cpu = 2
        self.x_pad = 3
        self.x_query = 10
        self.x_center = 60
        self.x_max = 65
        self.f0method = 'harvest'

vc = VC(RVCConfig())
vc.get_vc('shenhe_e250_s11250.pth')
print(f'✅ 申鹤模型加载完成 | 设备: {vc.config.device}')

In [ ]:
# 5. 测试推理
import numpy as np
import soundfile as sf

# 生成 1 秒测试音频
test_audio = np.random.randn(16000).astype(np.float32) * 0.01
sf.write('/tmp/test_input.wav', test_audio, 16000)

result = vc.vc_single(
    sid=0,
    input_audio_path='/tmp/test_input.wav',
    f0_up_key=0,
    f0_file=None,
    f0_method='harvest',
    file_index=f'{MODEL_DIR}/genshin_impact/shenhe_pitch.index',
    file_index2='',
    index_rate=0.5,
    filter_radius=3,
    resample_sr=0,
    rms_mix_rate=0.25,
    protect=0.33,
)

info, (sr, audio_out) = result
print(f'✅ 推理成功: {info[:100]}')
print(f'   输出采样率: {sr}Hz, 长度: {len(audio_out)/sr:.1f}s')

In [ ]:
# 6. 启动 API 服务
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS
import tempfile
import uuid
import base64

app = Flask(__name__)
CORS(app)

MODEL_NAME = 'shenhe_e250_s11250.pth'
INDEX_PATH = f'{MODEL_DIR}/genshin_impact/shenhe_pitch.index'

@app.route('/health')
def health():
    return jsonify({'status': 'ok', 'model': '申鹤 (Cinnamomo)', 'device': str(vc.config.device)})

@app.route('/tts', methods=['POST'])
def tts():
    data = request.json
    text = data.get('text', '')
    if not text:
        return jsonify({'error': 'text is required'}), 400

    # 1. 用 edge-tts 生成 TTS 音频
    import asyncio, edge_tts
    tts_path = f'/tmp/tts_{uuid.uuid4().hex[:8]}.mp3'
    
    async def gen_tts():
        comm = edge_tts.Communicate(text, 'zh-CN-XiaoxiaoNeural', rate='-10%')
        await comm.save(tts_path)
    asyncio.run(gen_tts())

    # 2. RVC 转换
    result = vc.vc_single(
        sid=0,
        input_audio_path=tts_path,
        f0_up_key=0,
        f0_file=None,
        f0_method='harvest',
        file_index=INDEX_PATH,
        file_index2='',
        index_rate=0.5,
        filter_radius=3,
        resample_sr=0,
        rms_mix_rate=0.25,
        protect=0.33,
    )

    info, (sr, audio) = result
    if audio is None:
        return jsonify({'error': f'RVC failed: {info}'}), 500

    # 3. 保存并返回
    out_path = f'/tmp/shenhe_{uuid.uuid4().hex[:8]}.wav'
    import soundfile as sf
    sf.write(out_path, audio.cpu().numpy() if hasattr(audio, 'cpu') else audio, sr)
    
    return send_file(out_path, mimetype='audio/wav', as_attachment=True,
                     download_name=f'shenhe_{uuid.uuid4().hex[:6]}.wav')

print('✅ Flask API 就绪')
print('   端点: /health (GET), /tts (POST)')
print('   启动 ngrok...')

In [ ]:
# 7. 启动 ngrok 隧道 + Flask
import threading
import time

def start_ngrok():
    time.sleep(3)
    !ngrok config add-authtoken 2gJmme4OOxo6rMJBPaHTvZT4i5k_4aueJmUXEC62SLrWUsRHP 2>/dev/null
    from pyngrok import ngrok
    tunnel = ngrok.connect(5000, 'http')
    print(f'\n🔗 公网 API: {tunnel.public_url}')
    print(f'   /health - 健康检查')
    print(f'   /tts   - POST {{"text": "你好"}} → 返回申鹤语音 wav')
    print(f'\n📋 复制此 URL 到本地 config.py 的 RVC_API_URL')

threading.Thread(target=start_ngrok).start()
app.run(host='0.0.0.0', port=5000)